# File 10 — Handcrafted Tongue Feature Extraction
Extracts tongue-only color/texture features from segmented images. No model training.


In [8]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from scipy.stats import entropy as sp_entropy

MANIFEST_PATH = Path(r'D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv')
OUTPUT_DIR = Path(r'D:\DIABETES\diabetes_pipeline_outputs\10_hybrid_image_features')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ERO_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
MIN_TONGUE_PIXELS = 100
print('Config loaded.')

def correct_lighting_clahe(image_pil, clip_limit=1.5, tile_grid_size=(8, 8)):
    """Deterministic CLAHE lighting correction. Matches File 11 training pipeline."""
    import numpy as np, cv2
    img_rgb = np.array(image_pil.convert('RGB'))
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq = clahe.apply(l)
    rgb_eq = cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2RGB)
    return Image.fromarray(rgb_eq)



Config loaded.


## Load Manifest


In [9]:
df = pd.read_csv(MANIFEST_PATH)
df = df[df['export_status'] == 'ok'].reset_index(drop=True)
img_col = 'segmented_image_path' if 'segmented_image_path' in df.columns else 'crop_masked_path'
df['image_path'] = df[img_col]
mask_col = 'predicted_mask_path' if 'predicted_mask_path' in df.columns else None
print(f'Loaded {len(df)} rows. Image col: {img_col}, Mask col: {mask_col}')
print(f'Splits: {df["final_split"].value_counts().to_dict()}')


Loaded 2750 rows. Image col: segmented_image_path, Mask col: predicted_mask_path
Splits: {'train': 1930, 'test': 412, 'val': 408}


## Feature Extraction Functions


In [10]:
def get_tongue_mask(img_rgb, mask_path=None):
    """Return binary tongue mask. Use predicted mask file or derive from non-black pixels."""
    if mask_path and Path(mask_path).exists():
        m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if m is not None:
            m = cv2.resize(m, (img_rgb.shape[1], img_rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
            return (m > 127).astype(np.uint8)
    # Fallback: non-black pixels
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    return (gray > 15).astype(np.uint8)


def erode_mask(mask, kernel=ERO_KERNEL):
    eroded = cv2.erode(mask, kernel, iterations=1)
    if eroded.sum() < MIN_TONGUE_PIXELS:
        return mask, True  # fallback
    return eroded, False


def color_ratio(hsv_pixels, h_lo, h_hi, s_lo, s_hi, v_lo, v_hi):
    h, s, v = hsv_pixels[:, 0], hsv_pixels[:, 1], hsv_pixels[:, 2]
    mask = (h >= h_lo) & (h <= h_hi) & (s >= s_lo) & (s <= s_hi) & (v >= v_lo) & (v <= v_hi)
    return mask.sum() / max(len(h), 1)


def extract_features(img_rgb, mask, eroded_mask):
    """Extract tongue-only color and texture features.
    CLAHE is applied to the image before feature computation.
    """
    import numpy as np
    # Apply CLAHE before feature extraction (matches File 11 CNN pipeline)
    img_pil_clahe = correct_lighting_clahe(Image.fromarray(img_rgb))
    img_rgb = np.array(img_pil_clahe)
    feat = {}
    tongue_px = img_rgb[mask == 1]
    if len(tongue_px) < MIN_TONGUE_PIXELS:
        return None

    # RGB stats
    for i, ch in enumerate(['r', 'g', 'b']):
        feat[f'rgb_mean_{ch}'] = float(tongue_px[:, i].mean())
        feat[f'rgb_std_{ch}'] = float(tongue_px[:, i].std())

    # HSV stats
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    hsv_px = hsv[mask == 1]
    for i, ch in enumerate(['h', 's', 'v']):
        feat[f'hsv_mean_{ch}'] = float(hsv_px[:, i].mean())
        feat[f'hsv_std_{ch}'] = float(hsv_px[:, i].std())

    # LAB stats
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    lab_px = lab[mask == 1]
    for i, ch in enumerate(['l', 'a', 'b']):
        feat[f'lab_mean_{ch}'] = float(lab_px[:, i].mean())
        feat[f'lab_std_{ch}'] = float(lab_px[:, i].std())

    # Color ratios (from HSV tongue pixels)
    feat['pink_red_ratio'] = color_ratio(hsv_px, 0, 15, 30, 255, 50, 255) + color_ratio(hsv_px, 160, 180, 30, 255, 50, 255)
    feat['white_coating_ratio'] = color_ratio(hsv_px, 0, 180, 0, 40, 160, 255)
    feat['yellow_ratio'] = color_ratio(hsv_px, 15, 35, 30, 255, 100, 255)
    feat['dark_tongue_pixel_ratio'] = float((hsv_px[:, 2] < 60).mean())
    feat['bright_tongue_pixel_ratio'] = float((hsv_px[:, 2] > 220).mean())
    feat['saturation_mean'] = float(hsv_px[:, 1].mean())
    feat['saturation_std'] = float(hsv_px[:, 1].std())
    feat['value_mean'] = float(hsv_px[:, 2].mean())
    feat['value_std'] = float(hsv_px[:, 2].std())

    # Texture features from eroded inner region
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    inner_px = gray[eroded_mask == 1]
    if len(inner_px) >= MIN_TONGUE_PIXELS:
        feat['laplacian_variance_inner'] = float(cv2.Laplacian(gray.astype(np.float64), cv2.CV_64F).var())
        # Compute on inner pixels only for entropy
        hist, _ = np.histogram(inner_px, bins=64, range=(0, 256), density=True)
        feat['entropy_inner'] = float(sp_entropy(hist + 1e-12))

        # GLCM on inner region bounding box
        iy, ix = np.where(eroded_mask == 1)
        patch = gray[iy.min():iy.max()+1, ix.min():ix.max()+1]
        patch_q = (patch // 4).astype(np.uint8)  # quantize to 64 levels
        try:
            glcm = graycomatrix(patch_q, distances=[1], angles=[0], levels=64, symmetric=True, normed=True)
            feat['glcm_contrast_inner'] = float(graycoprops(glcm, 'contrast')[0, 0])
            feat['glcm_homogeneity_inner'] = float(graycoprops(glcm, 'homogeneity')[0, 0])
            feat['glcm_energy_inner'] = float(graycoprops(glcm, 'energy')[0, 0])
            feat['glcm_correlation_inner'] = float(graycoprops(glcm, 'correlation')[0, 0])
        except Exception:
            feat['glcm_contrast_inner'] = feat['glcm_homogeneity_inner'] = 0.0
            feat['glcm_energy_inner'] = feat['glcm_correlation_inner'] = 0.0
    else:
        for k in ['laplacian_variance_inner','entropy_inner','glcm_contrast_inner',
                   'glcm_homogeneity_inner','glcm_energy_inner','glcm_correlation_inner']:
            feat[k] = 0.0

    return feat


def extract_qc_features(mask, img_shape):
    """QC-only features for reporting, NOT for the hybrid model."""
    h, w = img_shape[:2]
    fg = int(mask.sum())
    qc = {
        'qc_mask_foreground_ratio': fg / (h * w),
        'qc_blur_score': float(cv2.Laplacian(cv2.cvtColor(np.zeros((h, w, 3), dtype=np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64), cv2.CV_64F).var()),
    }
    if fg > 0:
        rows = np.any(mask, axis=1); cols = np.any(mask, axis=0)
        y1, y2 = np.where(rows)[0][[0, -1]]; x1, x2 = np.where(cols)[0][[0, -1]]
        qc['qc_bbox_area_ratio'] = ((x2-x1+1)*(y2-y1+1)) / (h*w)
        edge_count = sum([y1==0, y2==h-1, x1==0, x2==w-1])
        qc['qc_edge_touch'] = edge_count >= 3
        qc['qc_large_mask'] = qc['qc_mask_foreground_ratio'] > 0.85
        qc['qc_small_mask'] = qc['qc_mask_foreground_ratio'] < 0.02
    else:
        qc['qc_bbox_area_ratio'] = 0
        qc['qc_edge_touch'] = False
        qc['qc_large_mask'] = False
        qc['qc_small_mask'] = True
    return qc

print('Feature functions defined.')


Feature functions defined.


## Extract Features for All Images


In [11]:
records = []
failed = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='Extracting features'):
    try:
        img = cv2.imread(str(row['image_path']))
        if img is None:
            raise IOError('read failed')
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp = row.get(mask_col) if mask_col else None
        mask = get_tongue_mask(img_rgb, mp)
        eroded, erosion_fallback = erode_mask(mask)

        feat = extract_features(img_rgb, mask, eroded)
        if feat is None:
            failed.append({'image_id': row.get('image_id',''), 'reason': 'too_few_tongue_pixels', 'path': row['image_path']})
            continue

        qc = extract_qc_features(mask, img_rgb.shape)

        rec = {
            'image_id': row.get('image_id', ''),
            'final_split': row['final_split'],
            'final_label': row['final_label'],
            'label_binary': row['label_binary'],
            'segmented_image_path': row['image_path'],
            'predicted_mask_path': row.get(mask_col, '') if mask_col else '',
            'effective_group_id': row.get('effective_group_id', ''),
            'erosion_fallback': erosion_fallback,
        }
        rec.update(feat)
        rec.update(qc)
        records.append(rec)
    except Exception as e:
        failed.append({'image_id': row.get('image_id',''), 'reason': str(e), 'path': row['image_path']})

df_feat = pd.DataFrame(records)
print(f'Features extracted: {len(df_feat)}, Failed: {len(failed)}')


Extracting features: 100%|██████████| 2750/2750 [00:43<00:00, 62.87it/s] 

Features extracted: 2750, Failed: 0


## Save Outputs


In [12]:
# Feature matrix
df_feat.to_csv(OUTPUT_DIR / '10_feature_matrix.csv', index=False)
print(f'Saved: 10_feature_matrix.csv ({len(df_feat)} rows)')

# Column lists
MODEL_FEATURES = [
    'rgb_mean_r','rgb_mean_g','rgb_mean_b','rgb_std_r','rgb_std_g','rgb_std_b',
    'hsv_mean_h','hsv_mean_s','hsv_mean_v','hsv_std_h','hsv_std_s','hsv_std_v',
    'lab_mean_l','lab_mean_a','lab_mean_b','lab_std_l','lab_std_a','lab_std_b',
    'pink_red_ratio','white_coating_ratio','yellow_ratio',
    'dark_tongue_pixel_ratio','bright_tongue_pixel_ratio',
    'saturation_mean','saturation_std','value_mean','value_std',
    'laplacian_variance_inner','entropy_inner',
    'glcm_contrast_inner','glcm_homogeneity_inner','glcm_energy_inner','glcm_correlation_inner',
]

QC_FEATURES = ['qc_mask_foreground_ratio','qc_bbox_area_ratio','qc_edge_touch','qc_large_mask','qc_small_mask','qc_blur_score']

with open(OUTPUT_DIR / '10_model_feature_columns.json', 'w') as f:
    json.dump(MODEL_FEATURES, f, indent=2)
with open(OUTPUT_DIR / '10_qc_feature_columns.json', 'w') as f:
    json.dump(QC_FEATURES, f, indent=2)

# Failed rows
pd.DataFrame(failed).to_csv(OUTPUT_DIR / '10_feature_missing_or_failed_rows.csv', index=False)

# Summary
summary = df_feat.groupby('final_split').size().reset_index(name='count')
summary_cls = df_feat.groupby(['final_split','final_label']).size().reset_index(name='count')
summary_all = pd.concat([summary, summary_cls], ignore_index=True)
summary_all.to_csv(OUTPUT_DIR / '10_feature_extraction_summary.csv', index=False)

# Distribution summary
dist_rows = []
for col in MODEL_FEATURES:
    if col in df_feat.columns:
        for cls in [0, 1]:
            vals = df_feat[df_feat['label_binary']==cls][col].dropna()
            dist_rows.append({'feature':col,'class':cls,'mean':vals.mean(),'std':vals.std(),'min':vals.min(),'max':vals.max(),'median':vals.median()})
pd.DataFrame(dist_rows).to_csv(OUTPUT_DIR / '10_feature_distribution_summary.csv', index=False)

print('Summaries saved.')


Saved: 10_feature_matrix.csv (2750 rows)
Summaries saved.


In [13]:
# Correlation heatmap
avail = [c for c in MODEL_FEATURES if c in df_feat.columns]
corr = df_feat[avail].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, xticklabels=True, yticklabels=True, fmt='.1f', annot=False)
plt.xticks(fontsize=6, rotation=90); plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '10_feature_correlation_heatmap.png', dpi=100)
plt.close()

# Class boxplots — top features by mean difference
mean_diff = []
for c in avail:
    d = df_feat[df_feat['label_binary']==1][c].mean() - df_feat[df_feat['label_binary']==0][c].mean()
    mean_diff.append((c, abs(d)))
mean_diff.sort(key=lambda x: x[1], reverse=True)
top_feats = [x[0] for x in mean_diff[:12]]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, feat_name in enumerate(top_feats):
    ax = axes[i//4, i%4]
    for cls, color, label in [(0,'blue','non_diabetes'),(1,'red','diabetes')]:
        vals = df_feat[df_feat['label_binary']==cls][feat_name].dropna()
        ax.boxplot([vals], positions=[cls], widths=0.6, patch_artist=True,
                   boxprops=dict(facecolor=color, alpha=0.3))
    ax.set_title(feat_name, fontsize=7)
    ax.set_xticks([0,1]); ax.set_xticklabels(['ND','D'], fontsize=7)
for i in range(len(top_feats), 12):
    axes[i//4, i%4].axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '10_feature_class_boxplots.png', dpi=100)
plt.close()

print('Plots saved.')


Plots saved.


In [14]:
handoff = f"""FILE 10 HANDOFF — HANDCRAFTED FEATURE EXTRACTION
Status: {'PASS' if len(df_feat) > 0 else 'FAIL'}
Manifest: {MANIFEST_PATH}
Features extracted: {len(df_feat)}
Failed: {len(failed)}
Model features: {len(MODEL_FEATURES)}
QC features: {len(QC_FEATURES)} (NOT for model input)
Splits: {df_feat.groupby('final_split').size().to_dict()}
CLAHE used for handcrafted model features: YES
QC features excluded from classifier: YES
Segmentation geometry excluded from classifier: YES
Color from tongue pixels only: YES
Texture from eroded inner mask: YES
"""
with open(OUTPUT_DIR / '10_feature_handoff_summary.txt', 'w') as f:
    f.write(handoff)
print(handoff)


FILE 10 HANDOFF — HANDCRAFTED FEATURE EXTRACTION
Status: PASS
Manifest: D:\DIABETES\diabetes_pipeline_outputs\03_segmented_export_manifest.csv
Features extracted: 2750
Failed: 0
Model features: 33
QC features: 6 (NOT for model input)
Splits: {'test': 412, 'train': 1930, 'val': 408}
CLAHE used for handcrafted model features: YES
QC features excluded from classifier: YES
Segmentation geometry excluded from classifier: YES
Color from tongue pixels only: YES
Texture from eroded inner mask: YES

